In [1]:
# %pip install pylatexenc
# %pip install matplotlib
# %pip install git+https://github.com/It4innovations/quantum-as-a-service.git@main

In [2]:
import os

os.environ["QPROVIDER_LOGLEVEL"] = "DEBUG"

# Connect to system

In [ ]:
# # Setup Lexis session
from py4lexis.session import LexisSession
lexis_session = LexisSession(exit_on_error=True, aai_url="https://aai.test.lexis.tech", realm="LEXIS_AAI_v2_TEST", api_url="https://api.test.lexis.tech/")

In [ ]:
token = lexis_session.get_access_token()

In [ ]:
# token = 'eyJhb'

lexis_project = "vlq_demo_project"
resource_name = "VLQ-LEXIS-TEST-P0001"

- Using QProviderDev for development purposes (setup of queues is possible for this class)

In [6]:
from qaas.client.provider import QProviderDev
from qaas.client import QBackend, QJob

provider = QProviderDev(token, lexis_project)
# QBackend
backend: QBackend = provider.get_backend(
    resource_name,
    qinit_queue_name="init_queue",
    qexecute_queue_name="compute_queue",
    lexis_userorg_api_url="https://api.test.lexis.tech/userorg",
    lexis_aggregation_name=["VLQ-LEXIS-TEST"]
)

print(f"Qubit: {backend.architecture.qubits}")

print(f"Gates: {backend.architecture.gates.keys()}")

/Users/janswiatkowski/.pyenv/versions/qaas-v0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-27 18:16:17,654 - DEBUG - qaas.client.client - Detected JWT issuer: https://aai.test.lexis.tech/auth/realms/LEXIS_AAI_v2_TEST
2026-07-27 18:16:17,655 - DEBUG - qaas.client.client - JWKS: https://aai.test.lexis.tech/auth/realms/LEXIS_AAI_v2_TEST/protocol/openid-connect/certs
2026-07-27 18:16:18,042 - DEBUG - qaas.client.client - JWT token validated successfully for user: jan.swiatkowski@vsb.cz
2026-07-27 18:16:18,440 - DEBUG - qaas.client.client - Project access verified: vlq_demo_project
2026-07-27 18:16:18,441 - DEBUG - qaas.client.client - Project details: {'Id': 'ccc4dea0-d1d6-4a4c-bc71-3a46f1961c2a', 'Name': 'VLQ demo project', 'ShortName': 'vlq_demo_project', 'Description': 'VLQ demo/testing projec

Qubit: ['QB1', 'QB2', 'QB3', 'QB4', 'QB5', 'QB6', 'QB7', 'QB8', 'QB9', 'QB10', 'QB11', 'QB12', 'QB13', 'QB14', 'QB15', 'QB16', 'QB17', 'QB18', 'QB19', 'QB20', 'QB21', 'QB22', 'QB23', 'QB24']
Gates: dict_keys(['measure', 'measure_fidelity', 'prx', 'cz', 'move', 'reset_wait'])


# Define Quantum Circuit

In [7]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(15, 5)
for i in range(15):
    qc.h(i)
for i in range(14):
    qc.cx(i, i + 1)
qc.cx(0, 14)
qc.measure(range(5), range(5))

# Transpile Circuit

In [8]:
from qaas.client.backend_iqm import transpile_to_IQM

# Transpile circuit Locally
qc_transpiled_local = transpile_to_IQM(qc, backend, optimize_single_qubits=False)

# Run Circuit

In [9]:
from time import time

# Run circuit with number of SHOTS
SHOTS = 100

client_execution_time = time()

# Passing as argument not transpiled circuit
job: QJob = backend.run([qc_transpiled_local], shots=SHOTS)
result = job.result()
client_execution_time = time() - client_execution_time
results_dict = result.get_counts()
print("Raw counts:", results_dict)

# Print non-zero results
for key, count in results_dict.items():
    if count > 0:
        print(f"State '{key}': {count} counts")
        # Only convert to int if key is actually a binary string
        if all(c in "01" for c in key):
            print(f"  -> Decimal value: {int(key, 2)}")

2026-07-27 18:16:24,256 - DEBUG - qaas.client.backend - run_kwargs: {'shots': 100}
2026-07-27 18:16:24,257 - DEBUG - qaas.client.backend - len(circuit): 1
2026-07-27 18:16:24,258 - DEBUG - qaas.client.client - LEXIS_PROJECT_RESOURCE_ID: 019e92f8-a4c6-7966-bcd5-eccf17e933f4
2026-07-27 18:16:27,183 - DEBUG - qaas.client.client - Job '45' state '4', failed_code '32', False
2026-07-27 18:16:28,149 - DEBUG - qaas.client.client - Job '45' state '4', failed_code '32', False
2026-07-27 18:16:29,075 - DEBUG - qaas.client.client - Job '45' state '8', failed_code '32', False
2026-07-27 18:16:29,996 - DEBUG - qaas.client.client - Job '45' state '8', failed_code '32', False
2026-07-27 18:16:30,859 - DEBUG - qaas.client.client - Job '45' state '8', failed_code '32', False
2026-07-27 18:16:31,616 - DEBUG - qaas.client.client - Job '45' state '16', failed_code '32', False
2026-07-27 18:16:31,617 - DEBUG - qaas.client.backend - job finished in: 4.800097 s
2026-07-27 18:16:31,618 - DEBUG - qaas.client.c

Raw counts: {'00000': 100}
State '00000': 100 counts
  -> Decimal value: 0


In [10]:
print(f"Circuit runtime {job.remote_iqm_client_job_runtime} s")
print(f"Real HW execution {job.remote_hw_runtime} s")
print(f"Overall runtime on remote {job.remote_backend_runtime} s")
print("-" * 40)

print(f"QProvider runtime ended in {job.qaas_runtime} s")
print(f"QProvider fetched outputs in {job.qaas_fetching_runtime} s")
print(f"Pure client execution time {client_execution_time} s")

Circuit runtime 2.443474054336548 s
Real HW execution 0.829274 s
Overall runtime on remote 2.5427145957946777 s
----------------------------------------
QProvider runtime ended in 4.800097227096558 s
QProvider fetched outputs in 2.197496175765991 s
Pure client execution time 9.56519103050232 s


In [11]:
for entry in result.timeline:
    # if entry.status in ["execution_started", "execution_ended"]:
    print(entry)

source='iqm-server' status='created' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 29, 244891, tzinfo=TzInfo(0))
source='iqm-station-control' status='received' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 29, 284390, tzinfo=TzInfo(0))
source='iqm-station-control' status='validation_started' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 29, 339466, tzinfo=TzInfo(0))
source='iqm-station-control' status='validation_ended' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 29, 345067, tzinfo=TzInfo(0))
source='iqm-station-control' status='compilation_started' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 29, 361624, tzinfo=TzInfo(0))
source='iqm-station-control' status='compilation_ended' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 30, 81218, tzinfo=TzInfo(0))
source='iqm-station-control' status='execution_started' timestamp=datetime.datetime(2026, 7, 27, 16, 16, 30, 285058, tzinfo=TzInfo(0))
source='iqm-station-control' status='execution_ended' timestamp=datetime.datetime(

In [ ]:
from qiskit.visualization import plot_histogram

# Plot histogram
plot_histogram(results_dict, figsize=(12, 6), bar_labels=False)  # hide text above bars

## Run OpenQASM formatted circuit

In [ ]:
from qiskit.qasm3 import dumps as qasm3dumps

qasm_circuit_transpiled = qasm3dumps(qc_transpiled_local)
print(qasm_circuit_transpiled)

In [ ]:
# Run circuit with number of SHOTS
SHOTS = 100

# Passing as argument not transpiled circuit
qasm_job: QJob = backend.run([qasm_circuit_transpiled], shots=SHOTS)
qasm_result = qasm_job.result()
qasm_results_dict = qasm_result.get_counts()
print("Raw counts:", qasm_results_dict)

# Print non-zero results
for key, count in qasm_results_dict.items():
    if count > 0:
        print(f"State '{key}': {count} counts")
        # Only convert to int if key is actually a binary string
        if all(c in "01" for c in key):
            print(f"  -> Decimal value: {int(key, 2)}")